## 1. Configuration

In [ ]:
import glob
import os
import numpy as np
import astroalign as aa
from astropy.io import fits
from pyraf import iraf

# =====================================================================
# 1. USER CONFIGURATION
# =====================================================================

# ----------------- Align Configuration -----------------
input_wildcard_align = "aaGRB_R_*.fits"
output_prefix_align = "al_"

# ----------------- Imcombine Configuration -------------
input_wildcard_imcombine = f"{output_prefix_align}*.fits"
output_file_imcombine = "1st_R.fits"

# Detector characteristics (REQUIRED for ccdclip to work!)
detector_gain = "5.0"      # e-/ADU
detector_rdnoise = "10.0"   # e-
exptime_keyword = "EXPTIME" # FITS header keyword for exposure time

# Combination settings
combine_option = "average"  # 'average', 'sum', or 'median'
reject_option = "ccdclip"   # 'ccdclip' (best), 'avsigclip', or 'none'


## 2. Functions

In [2]:
def align_for_imcombine(file_pattern, output_prefix="aligned_"):
    """
    Aligns a set of FITS images to the first image in the sequence using astroalign.
    Saves the registered images to disk so they can be stacked by PyRAF's imcombine.
    """
    image_list = sorted(glob.glob(file_pattern))
    
    if len(image_list) < 2:
        print("[Alignment] WARNING: Need at least 2 images to align. Exiting.")
        return

    ref_image_path = image_list[0]
    print(f"\n--- Starting Alignment Pipeline ---")
    print(f"Reference image selected: {ref_image_path}")

    # =====================================================================
    # 1. LOAD & PREP REFERENCE IMAGE
    # =====================================================================
    with fits.open(ref_image_path) as hdul:
        # Force 64-bit float to prevent byte-swapping errors
        ref_data = hdul[0].data.astype(np.float64) 
        ref_header = hdul[0].header.copy()
        
    # Clean any NaN values 
    ref_data[np.isnan(ref_data)] = np.nanmedian(ref_data)
    
    # Crop the reference image for the calculation phase
    c_ref = ref_data[200:-200, 200:-200]

    # Save the reference image with the new prefix so it gets included in the stack
    ref_out_path = f"{output_prefix}{os.path.basename(ref_image_path)}"
    fits.writeto(ref_out_path, ref_data, ref_header, overwrite=True, output_verify='silentfix')
    print(f"[KEEP] {ref_image_path} -> {ref_out_path} (Master Reference)")

    successful_alignments = 1

    # =====================================================================
    # 2. ALIGN REMAINING IMAGES
    # =====================================================================
    for img_file in image_list[1:]:
        with fits.open(img_file) as hdul:
            source_data = hdul[0].data.astype(np.float64)
            source_header = hdul[0].header.copy()
            
        source_data[np.isnan(source_data)] = np.nanmedian(source_data)
        
        try:
            # Trick 1: Central Cropping to bypass edge artifacts (amplifier glow, vignetting)
            c_source = source_data[200:-200, 200:-200]
            
            # Trick 2: Bulletproof Astroalign parameters for faint fields
            t, _ = aa.find_transform(c_source, c_ref, min_area=9, detection_sigma=2.0)
            
            # Apply the transform to the FULL source image (not the cropped one!)
            # This ensures your final aligned image is the original size.
            registered_data, _ = aa.apply_transform(t, source_data, ref_data)
            
            # Save the aligned image with its original header
            out_path = f"{output_prefix}{os.path.basename(img_file)}"
            fits.writeto(out_path, registered_data, source_header, overwrite=True, output_verify='silentfix')
            
            successful_alignments += 1
            print(f"[KEEP] {img_file} -> {out_path} (Aligned successfully)")
            
        except Exception as e:
            # If clouds ruined the frame or it's too blurry to match stars, we drop it safely
            print(f"[REJECT] ⚠️ Could not align {img_file}: {e}")

    # =====================================================================
    # 3. SUMMARY
    # =====================================================================
    print(f"\n--- Alignment Complete ---")
    print(f"Successfully aligned {successful_alignments} out of {len(image_list)} images.")
    print(f"Ready for PyRAF: Run imcombine on '{output_prefix}*.fits'")

def run_imcombine(input_wildcard, output_file, combine_option, reject_option, detector_gain, detector_rdnoise, exptime_keyword):
    file_list = glob.glob(input_wildcard)
    N = len(file_list)

    print(f"Found {N} files matching '{input_wildcard}'.")
    print(f"Stacking using '{combine_option}' with '{reject_option}' rejection...")

    if N == 0:
        print("Error: No files found. Check your directory or wildcard pattern.")
        return

    # Load the immatch package which contains imcombine
    iraf.immatch()

    # RUN IMCOMBINE
    iraf.imcombine(
        input=input_wildcard,
        output=output_file,
        headers="", 
        bpmasks="", 
        rejmasks="", 
        nrejmasks="", 
        expmasks="", 
        sigmas="",
        imcmb="$I", 
        logfile="STDOUT",
        
        combine=combine_option, 
        reject=reject_option, 
        project="no", 
        outtype="real",
        outlimits="", 
        offsets="none", 
        masktype="none", 
        maskvalue=0, 
        blank=0.0,
        
        scale="exposure", 
        zero="mode", 
        weight="none", 
        statsec="", 
        expname=exptime_keyword,
        
        lthreshold="INDEF", 
        hthreshold="INDEF", 
        nlow=1, 
        nhigh=1, 
        nkeep=1,
        
        mclip="yes", 
        lsigma=3.0, 
        hsigma=3.0,
        
        rdnoise=detector_rdnoise, 
        gain=detector_gain, snoise=0.0,
        
        sigscale=0.1, 
        pclip=-0.5, 
        grow=0.0, 
        mode="al"
    )
    
    print("\nStacking complete. Calculating new noise model and updating headers...")

    # CALCULATE NEW EFFECTIVE GAIN & READ NOISE
    if combine_option == "average":
        gain_expr = f"({N} * gain)"
        rn_expr = f"(rdnoise / sqrt({N}))"
    elif combine_option == "sum":
        gain_expr = "gain"
        rn_expr = f"(rdnoise * sqrt({N}))"
    elif combine_option == "median":
        effective_N = (2.0 / 3.0) * N
        gain_expr = f"({effective_N} * gain)"
        rn_expr = f"(rdnoise / sqrt({effective_N}))"
    else:
        print("Unrecognized combine option! Skipping header update.")
        gain_expr, rn_expr = None, None

    # UPDATE THE OUTPUT FILE'S HEADER
    if gain_expr and rn_expr:
        iraf.hedit(images=output_file, fields="gain", value=gain_expr, add="yes", addonly="no", delete="no", verify="no", show="yes", update="yes")
        iraf.hedit(images=output_file, fields="rdnoise", value=rn_expr, add="yes", addonly="no", delete="no", verify="no", show="yes", update="yes")
        print("\nPipeline finished successfully! Your image is ready for DAOPHOT.")


## 3. Execute align

In [3]:
align_for_imcombine(
    file_pattern=input_wildcard_align,
    output_prefix=output_prefix_align
)


--- Starting Alignment Pipeline ---
Reference image selected: aaGRB_I_0.fits
[KEEP] aaGRB_I_0.fits -> al_aaGRB_I_0.fits (Master Reference)
[KEEP] aaGRB_I_1.fits -> al_aaGRB_I_1.fits (Aligned successfully)
[KEEP] aaGRB_I_2.fits -> al_aaGRB_I_2.fits (Aligned successfully)
[KEEP] aaGRB_I_3.fits -> al_aaGRB_I_3.fits (Aligned successfully)
[KEEP] aaGRB_I_4.fits -> al_aaGRB_I_4.fits (Aligned successfully)
[KEEP] aaGRB_I_5.fits -> al_aaGRB_I_5.fits (Aligned successfully)
[KEEP] aaGRB_I_6.fits -> al_aaGRB_I_6.fits (Aligned successfully)

--- Alignment Complete ---
Successfully aligned 7 out of 7 images.
Ready for PyRAF: Run imcombine on 'al_*.fits'


## 4. Execute imcombine

In [4]:
run_imcombine(
    input_wildcard=input_wildcard_imcombine,
    output_file=output_file_imcombine,
    combine_option=combine_option,
    reject_option=reject_option,
    detector_gain=detector_gain,
    detector_rdnoise=detector_rdnoise,
    exptime_keyword=exptime_keyword
)

Found 15 files matching 'al_*.fits'.
Stacking using 'average' with 'ccdclip' rejection...



Stacking complete. Calculating new noise model and updating headers...
1st_I.fits,gain: 60 -> 900
1st_I.fits updated
1st_I.fits,rdnoise: 2.886751 -> 0.7453558
1st_I.fits updated

Pipeline finished successfully! Your image is ready for DAOPHOT.
